## RAG for small-scale Agents

In [ ]:
## Tools are interfaces that an agent, chain, or LLM can use to interact with the world. They can be used to perform actions, retrieve information, or manipulate data to perform specific tasks

## WikipediaQueryRun - Tool that searches the Wikipedia API for a given query. It returns a list of search results to retrieve more information about a specific topic
## WikipediaAPIWrapper - This wrapper uses the Wikipedia API to conduct searches and fetch page summaries. By default, it will return the page summaries of the top-k results
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

### 1. Wikipedia API tool

In [ ]:
## ANALOGY: WikipediaAPIWrapper = settings; WikipediaQueryRun = search tool
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)     # WikipediaQueryRun(...) is the tool class that actually queries Wikipedia, api_wrapper=api_wrapper - passes in the Wikipedia settings object. This wrapper controls things like how many results to return and how much text to keep

In [3]:
wiki.name

'wikipedia'

### 2. ollama webpage retriever tool

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
loader=WebBaseLoader("https://github.com/ollama/ollama")
docs=loader.load()          # entire content from the webpage is loaded as a single document in a list. The loader fetches the content of the specified URL and returns it as a list of documents, where each document is typically a string containing the text content of the webpage
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)   # split the loaded documents as per specified parameters
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")   # initialize the embedding model for generating vector representations of text data
db = FAISS.from_documents(documents[:20],embeddings)
retriever=db.as_retriever()
retriever

C:\Users\nisha\AppData\Local\Temp\ipykernel_13380\784534850.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")   # initialize the embedding model to be used for generating vector representations of text data.


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E0A3EDEDA0>, search_kwargs={})

In [ ]:
## create_retriever_tool is a helper that turns a retriever into a tool an agent can call. In short: it wraps search/retrieval into something the LLM can use like a normal tool
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"ollama_search",
                      "Search for information about Ollama. For any questions about Ollama, you must use this tool!")

# The retriever tool is a tool that allows us to search for information about Ollama. It uses the retriever object that we created earlier to perform the search. The name of the tool is "ollama_search" and it has a description that explains its purpose

In [7]:
retriever_tool.name

'ollama_search'

### 3. Arxiv Tool

In [ ]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

In [9]:
arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [10]:
# combining all the tools together
tools=[wiki,arxiv,retriever_tool]

In [11]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\All_Work\\LangChainProj\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='ollama_search', description='Search for information about Ollama. For any questions about Ollama, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001E0A3EEAD40>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001E0A4065480>)]

### Creating Agents

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",                        # LM Studio's local server URL
    api_key="lm-studio",                                        # LM Studio doesn't need a real key, any string works
    model="qwen3.5-4b-claude-4.6-opus-reasoning-distilled",     # API Model identifier in LM Studio
    temperature=0.8,
)

In [ ]:
## Agents - to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order
## When we give any query, our agent is going to check each of the tool in order and decide which tool to use based on the query. If the agent needs more information, it can use another tool. This process continues until the agent has enough information to answer the query
## ReAct Loop - The agent follows an iterative cycle: Reasoning (the LLM decides what to do) → Action (calling a tool) → Observation (viewing the tool result). 2. Persistence: We can add a checkpointer (like MemorySaver) during graph compilation to enable the agent to remember past interactions across different threads. 3. Control Flow: Unlike traditional linear chains, LangGraph allows for cyclic graphs, which is essential for ReAct agents that may need multiple tool steps to solve a problem
from langchain.agents import create_agent
agent = create_agent(llm, tools)            # uses it's own built in prompt

In [ ]:
## using the Ollama webpage retriever tool
 
agent.invoke({"messages": [("user", "Tell me about Ollama")]})

{'messages': [HumanMessage(content='Tell me about Ollama', additional_kwargs={}, response_metadata={}, id='a9c81de7-be81-424c-b3f3-e6e06b45851d'),
  AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 398, 'total_tokens': 510, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 84, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3.5-4b-claude-4.6-opus-reasoning-distilled', 'system_fingerprint': 'qwen3.5-4b-claude-4.6-opus-reasoning-distilled', 'id': 'chatcmpl-j89sb4dwk7fwugqt67c0h', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8ba1-f4fe-7c32-9c7c-a3d64065778a-0', tool_calls=[{'name': 'ollama_search', 'args': {'query': 'What is Ollama'}, 'id': '363692441', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 398, 'output_tokens': 11

In [ ]:
## using the Arxiv tool to search for a paper

agent.invoke({"messages": [("user", "What is the paper 2103.00020 about ?")]})

{'messages': [HumanMessage(content='What is the paper 2103.00020 about ?', additional_kwargs={}, response_metadata={}, id='6d8e22d7-c342-42cf-a90b-acc1e98b2cee'),
  AIMessage(content='\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 409, 'total_tokens': 548, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 107, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3.5-4b-claude-4.6-opus-reasoning-distilled', 'system_fingerprint': 'qwen3.5-4b-claude-4.6-opus-reasoning-distilled', 'id': 'chatcmpl-2bk7h16k8fa53vpg0fde3j', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d8baa-e833-7900-8f33-c91b584da297-0', tool_calls=[{'name': 'arxiv', 'args': {'query': '2103.00020'}, 'id': '119134148', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 409, 'output_token

In [ ]:
## using the Wikipedia tool to search for a topic

agent.invoke({"messages": [("user", "Tell me about Machine Learning")]})

{'messages': [HumanMessage(content='Tell me about Machine Learning', additional_kwargs={}, response_metadata={}, id='fdd76392-d20a-4175-8581-b29981a47311'),
  AIMessage(content='\n\n# Machine Learning: A Comprehensive Overview\n\nMachine Learning (ML) is a transformative field of artificial intelligence that enables systems to learn from data, make decisions, and improve over time without being explicitly programmed for every task. Let me break down this fascinating subject:\n\n## What is Machine Learning?\n\n**Definition:** ML is the scientific study of algorithms and statistical models that computer systems use to perform tasks such as prediction, classification, clustering, and regression—tasks that typically require human intelligence but are achieved through learning from data rather than explicit programming.\n\n---\n\n## Historical Timeline\n\n| Era | Key Developments |\n|-----|------------------|\n| **1940s-50s** | Turing\'s "Computing Machinery and Intelligence" (1950); Percep

In [ ]:
## streaming responses - stream_mode="updates" -gives updates as the agent thinks through the problem and calls tools, instead of waiting for the final answer. This allows us to see the intermediate steps the agent takes

for step in agent.stream(
    {"messages": [("user", "Tell me about Langsmith")]},
    stream_mode="updates"
):
    for key, value in step.items():
        if key == "tools":
            print(f"Tool used: {value['messages'][0].name}")
        elif key == "agent":
            print(f"Agent response: {value['messages'][0].content}")

Tool used: ollama_search
Tool used: wikipedia
Tool used: wikipedia
Tool used: wikipedia
Tool used: ollama_search
Tool used: wikipedia
Tool used: wikipedia
Tool used: arxiv
Tool used: wikipedia
Tool used: wikipedia
Tool used: wikipedia
Tool used: wikipedia
Tool used: arxiv
Tool used: wikipedia
